# **Modelos Referenciales**
### Proyecto Hito 1
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- User KNN](#2--user-knn)

>[3- Item KNN](#3--item-knn)

>[4- Most Popular](#4--most-popular)

>[5- Random](#5--random)

## 0- Instalación de librerías

In [1]:
# !pip uninstall -y numpy
# !pip install numpy==1.26

In [2]:
# !pip install scikit-surprise --no-build-isolation --no-deps

In [3]:
# pip install pandas

## 1- Carga de datos

In [4]:
import surprise
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
import random

Se lee el archivo de datos y se almacena en un dataframe:

In [5]:
df = pd.read_csv('video_game_reviews.csv')

Se añade la ID de usuario para cada evaluación del dataframe, ya que no poseen. Para ello, se generan 3000 ID de usuarios (del 1 al 3000) que se reparten aleatoriamente por las evaluaciones realizadas. Además, se fija la semilla del random para que sea replicable. Se agrega la columna "user_id" al dataframe:

In [6]:
np.random.seed(42)
df['user_id'] = np.random.randint(1, 3001, size=len(df))

Se añade ID de ítem, ya que tampoco se posee. Para eso, se asigna arbitrariamente una ID a cada videojuego que aparece en el set de datos. Ya que hay 40 videojuegos diferentes, se asignan ID desde el 1 al 40:

In [7]:
print(f'Videojuegos unicos: {df["Game Title"].unique()}')

Videojuegos unicos: ['Grand Theft Auto V' 'The Sims 4' 'Minecraft' 'Bioshock Infinite'
 'Half-Life: Alyx' 'Sid Meier’s Civilization VI' 'Just Dance 2024'
 '1000-Piece Puzzle' 'Spelunky 2' 'Street Fighter V' 'Fall Guys'
 'Rocket League' 'The Elder Scrolls V: Skyrim' 'Among Us' 'Stardew Valley'
 'Call of Duty: Modern Warfare 2'
 'The Legend of Zelda: Breath of the Wild' 'Tekken 7'
 'Pillars of Eternity II: Deadfire' 'Animal Crossing: New Horizons'
 'Hades' 'Mario Kart 8 Deluxe' 'Overwatch 2' 'Fortnite'
 'Pokémon Scarlet & Violet' 'Hitman 3' 'Tomb Raider (2013)'
 'Halo Infinite' 'Super Smash Bros. Ultimate' 'Kingdom Hearts III'
 'League of Legends' 'The Witcher 3: Wild Hunt' 'FIFA 24'
 'Ghost of Tsushima' 'Cuphead' 'Red Dead Redemption 2' 'Portal 2' 'Tetris'
 'Counter-Strike: Global Offensive' 'Super Mario Odyssey']


Se añade la columna "item_id" al dataframe:

In [8]:
df['item_id'] = df['Game Title'].astype('category').cat.codes + 1

In [9]:
df

,Game Title,User Rating,Age Group Targeted,Price,Platform,Requires Special Device,Developer,Publisher,Release Year,Genre,Multiplayer,Game Length (Hours),Graphics Quality,Soundtrack Quality,Story Quality,User Review Text,Game Mode,Min Number of Players,user_id,item_id
0,Grand Theft Auto V,36.4,All Ages,41.41,PC,No,Game Freak,Innersloth,2015,Adventure,No,55.3,Medium,Average,Poor,"Solid game, but too many bugs.",Offline,1,861,12
1,The Sims 4,38.3,Adults,57.56,PC,No,Nintendo,Electronic Arts,2015,Shooter,Yes,34.6,Low,Poor,Poor,"Solid game, but too many bugs.",Offline,3,1295,38
2,Minecraft,26.8,Teens,44.93,PC,Yes,Bungie,Capcom,2012,Adventure,Yes,13.9,Low,Good,Average,"Great game, but the graphics could be better.",Offline,5,1131,21
3,Bioshock Infinite,38.4,All Ages,48.29,Mobile,Yes,Game Freak,Nintendo,2015,Sports,No,41.9,Medium,Good,Excellent,"Solid game, but the graphics could be better.",Online,4,1096,4
4,Half-Life: Alyx,30.1,Adults,55.49,PlayStation,Yes,Game Freak,Epic Games,2022,RPG,Yes,13.2,High,Poor,Good,"Great game, but too many bugs.",Offline,1,1639,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47769,Minecraft,41.6,All Ages,49.11,PlayStation,Yes,Valve,Innersloth,2016,Party,No,56.7,Low,Excellent,Average,"Amazing game, but the graphics could be better.",Online,4,1293,21
47770,The Legend of Zelda: Breath of the Wild,24.2,Kids,31.69,Xbox,Yes,Valve,Electronic Arts,2023,Strategy,No,29.7,High,Good,Excellent,"Solid game, but the graphics could be better.",Offline,1,2485,37
47771,Animal Crossing: New Horizons,26.7,All Ages,44.90,PlayStation,Yes,Game Freak,Square Enix,2018,Sports,No,8.2,Low,Poor,Good,"Great game, but the gameplay is amazing.",Offline,5,2675,3
47772,The Legend of Zelda: Breath of the Wild,22.5,Kids,29.99,Xbox,Yes,Epic Games,Epic Games,2018,Simulation,Yes,29.7,High,Poor,Excellent,"Disappointing game, but the graphics could be ...",Offline,1,2599,37


Se define un diccionario para almacenar el ttítulo del videojuego a partir de su ID, para análisis posterior:

In [10]:
info_videojuegos = dict(zip(df['item_id'], zip(df['Game Title'], df['Genre'])))
info_videojuegos = {str(k): v for k, v in info_videojuegos.items()}

In [11]:
info_videojuegos

{'12': ('Grand Theft Auto V', 'Adventure'),
 '38': ('The Sims 4', 'Simulation'),
 '21': ('Minecraft', 'Party'),
 '4': ('Bioshock Infinite', 'Sports'),
 '14': ('Half-Life: Alyx', 'Strategy'),
 '28': ('Sid Meier’s Civilization VI', 'Party'),
 '17': ('Just Dance 2024', 'Party'),
 '1': ('1000-Piece Puzzle', 'Fighting'),
 '29': ('Spelunky 2', 'RPG'),
 '31': ('Street Fighter V', 'Strategy'),
 '9': ('Fall Guys', 'Adventure'),
 '27': ('Rocket League', 'Puzzle'),
 '36': ('The Elder Scrolls V: Skyrim', 'Sports'),
 '2': ('Among Us', 'Sports'),
 '30': ('Stardew Valley', 'Strategy'),
 '5': ('Call of Duty: Modern Warfare 2', 'RPG'),
 '37': ('The Legend of Zelda: Breath of the Wild', 'Sports'),
 '34': ('Tekken 7', 'Action'),
 '23': ('Pillars of Eternity II: Deadfire', 'Shooter'),
 '3': ('Animal Crossing: New Horizons', 'Sports'),
 '13': ('Hades', 'Puzzle'),
 '20': ('Mario Kart 8 Deluxe', 'Action'),
 '22': ('Overwatch 2', 'Fighting'),
 '10': ('Fortnite', 'Strategy'),
 '24': ('Pokémon Scarlet & Violet'

In [12]:
all_items = set(info_videojuegos.keys())

In [13]:
all_items

{'1',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '3',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '4',
 '40',
 '5',
 '6',
 '7',
 '8',
 '9'}

Reordenar columnas:

In [14]:
cols = df.columns.tolist()
new_order = ['user_id', 'item_id'] + [c for c in cols if c not in ['user_id', 'item_id']]
df = df[new_order]

Guardar dataframe reordenado y con las ID de usuario e ítem agregadas en un nuevo archivo .csv:

In [15]:
df.to_csv('video_game_reviews_with_userid.csv', index=False)

Reviews por usuario:

In [16]:
print(df['user_id'].value_counts())

user_id
2839    32
2135    30
948     30
2774    30
993     29
        ..
544      5
1581     5
2050     5
2465     5
424      5
Name: count, Length: 3000, dtype: int64


Verificar cantidad de usuarios:

In [17]:
print(f"Numero de usuarios unicos: {df['user_id'].nunique()}")

Numero de usuarios unicos: 3000


Se lee el último .csv y se guardan solamente las columnas de ID de usuario, ID de ítem, y el rating correspondiente:

In [18]:
df = pd.read_csv('video_game_reviews_with_userid.csv', sep=',')
df = df[['user_id', 'item_id', 'User Rating']]

Convertir ratings a escala del 1 al 5:

In [19]:
def parametrizar_rating_a_5(df):
    vals = df["User Rating"].to_numpy().astype(float)

    minimo = np.min(vals)
    maximo = np.max(vals)

    scaled = 1 + ( (vals - minimo) / (maximo - minimo) ) * (5 - 1)

    df["rating"] = scaled

    return df

Se usa la función en el dataframe para generar la nueva columna "rating", con el valor del rating reescalado del 1 al 5, y se elimina la antigua columna "User Rating":

In [20]:
df = parametrizar_rating_a_5(df)
df.drop("User Rating", axis=1, inplace=True)

El dataframe final que se utilizará es:

In [21]:
df

,user_id,item_id,rating
0,861,12,3.670051
1,1295,38,3.862944
2,1131,21,2.695431
3,1096,4,3.873096
4,1639,14,3.030457
...,...,...,...
47769,1293,21,4.197970
47770,2485,37,2.431472
47771,2675,3,2.685279
47772,2599,37,2.258883


Se guarda el dataframe modificado en un nuevo archivo .csv:

In [22]:
df.to_csv('video_game_reviews_with_userid_clean.csv', index=False)

Se lee el .csv con los datos modificados y se definen los datasets de entrenamiento y testeo:

In [23]:
reader = Reader(line_format='user item rating', sep=',', rating_scale=(1,5), skip_lines=1)
data = Dataset.load_from_file('video_game_reviews_with_userid_clean.csv', reader=reader)

trainset, testset = train_test_split(data, test_size=0.2)

print("Usuarios:", trainset.n_users, "Items:", trainset.n_items, "Test size:", len(testset))

Usuarios: 3000 Items: 40 Test size: 9555


## 2- User KNN

Se busca el mejor valor de 'k' entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de 'k' y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [24]:
# código de Felipe Fuentes y Nicolás Bueno, usado en su tarea

#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': True})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Comput

Para realizar la recomendación, se define el modelo User KNN con similitud de coseno y k = 100, que es la combinación que alcanzó menor RMSE:

In [25]:
myUserKnn = surprise.KNNBasic(k=100, sim_options={'name': 'cosine', 'user_based': True})

In [26]:
myUserKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 1:

In [27]:
myUserKnn.predict("457", "1")

Prediction(uid='457', iid='1', r_ui=None, est=3.0522570155596944, details={'actual_k': 100, 'was_impossible': False})

Se realizan las predicciones a partir del antitest set:

In [28]:
a_testset = trainset.build_anti_testset()
predictions = myUserKnn.test(a_testset)

### Definición de métricas para evaluación

Las métricas están basadas en los códigos entregados en los prácticos. A continuación, se definen las que usaremos.

In [29]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0: return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0: return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if not rec_k: return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

Ahora, la idea fue definir una función que calcula el Top-K recomendado para cada usuario, ordenado por el score estimado. Está fuertemente basada en la get_top_n de los prácticos, pero adaptada a nuestras necesidades. Toma como argumentos las predicciones, que está en el formato Surprise, k, un booleano de unico_por_item (que si es True va a elimiar los duplicados de iids, dejando solo el de mayor score estimado), y devuelve un diccionario con los usuarios como keys y una lista de tuplas (item_id, score_estimado) como valores.

In [30]:
def topk_por_usuario(predictions, k=10, unico_por_item=True):
    # 1) agrupar por usuario
    user_scores = defaultdict(list)
    for p in predictions:
        # soporta tanto tupla como objeto prediction
        try:
            uid, iid, _, est, _ = p # esta pensado para una tupla
        except Exception:
            uid, iid, est = p.uid, p.iid, p.est # o bien objeto Prediction
        user_scores[uid].append((iid, est))

    # 2) ordenar desc por score y borrar duplicados opcionalmente
    topk = {}
    for uid, items in user_scores.items():
        items.sort(key=lambda x: x[1], reverse=True)
        if unico_por_item:
            seen = set()
            ordered_unique = []
            for it, sc in items:
                if it not in seen:
                    seen.add(it)
                    ordered_unique.append((it, sc))
            items = ordered_unique

        # 3) recorte a K y formato de salida
        items = items[:k]
        topk[uid] = [it for it, _ in items]

    return topk


También, definimos los elementos relevantes del testset, que serán aquellos con rating >= 4.

In [31]:
REL_THRESHOLD = 4.0
reales = defaultdict(set)
for (uid, iid, r) in testset:
    if r >= REL_THRESHOLD:
        reales[uid].add(iid)

La idea también es evaluar a solo usuarios comunes entre los que tienen predicciones y los que tienen datos reales en el testset. Así, procedemos:

In [32]:
# top-K unico por usuario
user_topk = topk_por_usuario(predictions, k=10, unico_por_item=True)

# evaluaa solo usuarios comunes
usuarios_comunes = set(reales.keys()) & set(user_topk.keys())
user_topk_eval = {u: user_topk[u] for u in usuarios_comunes}

# métricas agregadas
k = 10
diversidad_k = diversity_at_k(user_topk_eval, info_videojuegos)

precs, recs, ndcgs, hits, maps = [], [], [], [], []
for u in usuarios_comunes:
    rec_k = user_topk[u]
    rel_u = reales[u]
    precs.append(precision_at_k(rec_k, rel_u))
    recs.append(recall_at_k(rec_k, rel_u))
    ndcgs.append(ndcg_at_k(rec_k, rel_u))
    hits.append(hit_score_at_k(rec_k, rel_u))
    maps.append(map_at_k(rec_k, rel_u))

result_userknn = pd.DataFrame([{
    "Modelo": "UserKNN",
    f"Precision@{k}": float(np.mean(precs)) if precs else np.nan,
    f"Recall@{k}": float(np.mean(recs)) if recs else np.nan,
    f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else np.nan,
    f"HitScore@{k}": float(np.mean(hits)) if hits else np.nan,
    f"F1@{k}": (2 * np.mean(precs) * np.mean(recs) / (np.mean(precs) + np.mean(recs))) if precs and recs and (np.mean(precs) + np.mean(recs)) > 0 else np.nan,
    f"MAP@{k}": float(np.mean(maps)) if maps else np.nan,
    f"Diversidad@{k}": diversidad_k,
    "Usuarios evaluados": len(usuarios_comunes)
}])

result_userknn


,Modelo,Precision@10,Recall@10,NDCG@10,HitScore@10,F1@10,MAP@10,Diversidad@10,Usuarios evaluados
0,UserKNN,0.027718,0.238152,0.110364,0.266428,0.049657,0.068344,6.784946,837


Se obtiene la lista de recomendación top 10 para cada usuario y s emuestra como ejemplo el caso del con ID = 457:

In [33]:
print(user_topk["457"])

['22', '8', '29', '9', '1', '7', '33', '19', '10', '17']


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su rating predicho:

In [34]:
for item_id in user_topk["457"]:
    print(f"{info_videojuegos[str(item_id)]}")

('Overwatch 2', 'Fighting')
('FIFA 24', 'Shooter')
('Spelunky 2', 'RPG')
('Fall Guys', 'Adventure')
('1000-Piece Puzzle', 'Fighting')
('Cuphead', 'RPG')
('Super Smash Bros. Ultimate', 'RPG')
('League of Legends', 'Shooter')
('Fortnite', 'Strategy')
('Just Dance 2024', 'Party')


## 3- Item KNN

Se busca el mejor valor de K entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de K y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [35]:
#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': False})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Comput

Para realizar la recomendación, se define el modelo Item KNN con similitud de coseno y k = 30, que es la combinación que alcanzó menor RMSE:

In [36]:
myItemKnn = surprise.KNNBasic(k=30, sim_options={'name': 'cosine', 'user_based': False})

In [37]:
myItemKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 1:

In [38]:
myItemKnn.predict("457", "1")

Prediction(uid='457', iid='1', r_ui=None, est=2.7297189862608087, details={'actual_k': 15, 'was_impossible': False})

Se realizan las predicciones a partir del antitest set definido en la sección anterior:

In [39]:
predictions_item = myItemKnn.test(a_testset)

In [40]:
# top-K unico por usuario
user_topk = topk_por_usuario(predictions_item, k=10, unico_por_item=True)

# evaluaa solo usuarios comunes
usuarios_comunes = set(reales.keys()) & set(user_topk.keys())
user_topk_eval = {u: user_topk[u] for u in usuarios_comunes}

# métricas agregadas
k = 10
diversidad_k = diversity_at_k(user_topk_eval, info_videojuegos)

precs, recs, ndcgs, hits, maps = [], [], [], [], []
for u in usuarios_comunes:
    rec_k = user_topk[u]
    rel_u = reales[u]
    precs.append(precision_at_k(rec_k, rel_u))
    recs.append(recall_at_k(rec_k, rel_u))
    ndcgs.append(ndcg_at_k(rec_k, rel_u))
    hits.append(hit_score_at_k(rec_k, rel_u))
    maps.append(map_at_k(rec_k, rel_u))

result_itemknn = pd.DataFrame([{
    "Modelo": "UserKNN",
    f"Precision@{k}": float(np.mean(precs)) if precs else np.nan,
    f"Recall@{k}": float(np.mean(recs)) if recs else np.nan,
    f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else np.nan,
    f"HitScore@{k}": float(np.mean(hits)) if hits else np.nan,
    f"F1@{k}": (2 * np.mean(precs) * np.mean(recs) / (np.mean(precs) + np.mean(recs))) if precs and recs and (np.mean(precs) + np.mean(recs)) > 0 else np.nan,
    f"MAP@{k}": float(np.mean(maps)) if maps else np.nan,
    f"Diversidad@{k}": diversidad_k,
    "Usuarios evaluados": len(usuarios_comunes)
}])

result_itemknn


,Modelo,Precision@10,Recall@10,NDCG@10,HitScore@10,F1@10,MAP@10,Diversidad@10,Usuarios evaluados
0,UserKNN,0.029869,0.248507,0.116083,0.285544,0.053328,0.071288,6.700119,837


Se obtiene la lista de recomendación top 10 para cada usuario (usando get_top_n() definida arriba) y se muestra como ejemplo el caso del con ID = 457:

In [41]:
print(user_topk["457"])

['1', '4', '40', '10', '32', '39', '28', '17', '2', '19']


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su rating predicho:

In [42]:
for item_id in user_topk["457"]: 
    print(f"{info_videojuegos[str(item_id)][0]}")

1000-Piece Puzzle
Bioshock Infinite
Tomb Raider (2013)
Fortnite
Super Mario Odyssey
The Witcher 3: Wild Hunt
Sid Meier’s Civilization VI
Just Dance 2024
Among Us
League of Legends


## 4- Most Popular

Se realiza el procedimiento para recomendar a cada usuario los 10 videojuegos más populares que no haya visto. Para ello, se calcula para cada videojuego un 'score' de popularidad, que es la suma de todos los rating con el que los usuarios le han evaluado. Luego, los videojuegos más populares serán aquellos con el score más alto. Se define una función que realiza la recomendación top N para cada usuario:

In [43]:
def get_top_n_most_popular(trainset, n=10):

    # se calcula el score de popularidad de cada item en el conjunto de entrenamiento
    item_popularity = defaultdict(int)
    for uid, iid, rating in trainset.all_ratings():
        item_popularity[iid] += rating

    # se ordenan los items por popularidad
    popular_items = sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)

    # diccionario para las recomendaciones
    top_n = defaultdict(list)

    # para cada usuario, se recomiendan los n items con mayor socre de popularidad y que no haya visto
    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        count = 0
        for iid, _ in popular_items:
            if iid not in user_items:
                top_n[trainset.to_raw_uid(uid)].append((trainset.to_raw_iid(iid), item_popularity[iid]))
                count += 1
                if count >= n:
                    break

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457. En la lista de recomendación, el primer elemento de cada tupla es la ID del ítem, y el segundo es su score de popularidad:

In [44]:
top_n_most_popular = get_top_n_most_popular(trainset, n=10)
print(top_n_most_popular["457"])

[('29', 2982.142131979695), ('8', 2973.3959390862947), ('40', 2952.741116751264), ('33', 2943.5837563451837), ('1', 2937.157360406091), ('22', 2899.5482233502535), ('39', 2896.837563451778), ('9', 2892.7258883248746), ('3', 2884.69543147208), ('23', 2860.893401015228)]


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su score de popularidad:

In [45]:
for item_id, score in top_n_most_popular["457"]:
    print(f"{info_videojuegos[item_id][0]}")

Spelunky 2
FIFA 24
Tomb Raider (2013)
Super Smash Bros. Ultimate
1000-Piece Puzzle
Overwatch 2
The Witcher 3: Wild Hunt
Fall Guys
Animal Crossing: New Horizons
Pillars of Eternity II: Deadfire


Muy basado en todo lo anterior, tenemos la función para evaluar las métricas de Most Popular:

In [46]:
def evaluate_most_popular_metrics(top_n_most_popular, k=10, reales=reales):
    # top-k por usuario
    user_topk = {uid: [iid for (iid, _) in pairs[:k]] for uid, pairs in top_n_most_popular.items()}

    usuarios_comunes = set(reales.keys()) & set(user_topk.keys())

    precs, recs, ndcgs, hits, maps = [], [], [], [], []
    for u in usuarios_comunes:
        rec_k = user_topk[u]
        rel_u = reales[u]
        precs.append(precision_at_k(rec_k, rel_u))
        recs.append(recall_at_k(rec_k, rel_u))
        ndcgs.append(ndcg_at_k(rec_k, rel_u))
        hits.append(hit_score_at_k(rec_k, rel_u))
        maps.append(map_at_k(rec_k, rel_u))
    
    user_topk_eval = {u: user_topk[u] for u in usuarios_comunes}
    diversidad_k = diversity_at_k(user_topk_eval, info_videojuegos)

    return pd.DataFrame([{
        "Modelo": "MostPopular",
        f"Precision@{k}": float(np.mean(precs)) if precs else np.nan,
        f"Recall@{k}": float(np.mean(recs)) if recs else np.nan,
        f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else np.nan,
        f"HitScore@{k}": float(np.mean(hits)) if hits else np.nan,
        f"F1@{k}": (2 * np.mean(precs) * np.mean(recs) / (np.mean(precs) + np.mean(recs))) if precs and recs and (np.mean(precs) + np.mean(recs)) > 0 else np.nan,
        f"MAP@{k}": float(np.mean(maps)) if maps else np.nan,
        f"Diversidad@{k}": diversidad_k,
        "Usuarios evaluados": len(usuarios_comunes)
    }])


In [47]:
result_mostpopular = evaluate_most_popular_metrics(
    top_n_most_popular=top_n_most_popular,
    k=10,
    reales=reales
)

display(result_mostpopular)


,Modelo,Precision@10,Recall@10,NDCG@10,HitScore@10,F1@10,MAP@10,Diversidad@10,Usuarios evaluados
0,MostPopular,0.029271,0.244325,0.124062,0.27718,0.052279,0.082532,6.58184,837


## 5- Random

Se realiza el procedimiento para recomendar a cada usuario 10 videojuegos aleatorios que no haya visto. Se define una función que realiza la recomendación top N para cada usuario, utilizando una seed de random para que sea replicable:

In [48]:
import random

def get_top_n_random(trainset, n=10):
    random.seed(42)
    all_items = set(iid for iid in range(trainset.n_items))
    top_n = defaultdict(list)

    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        available_items = list(all_items - user_items)
        random_items = random.sample(available_items, min(n, len(available_items)))
        top_n[trainset.to_raw_uid(uid)] = [(trainset.to_raw_iid(iid)) for iid in random_items]

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457:

In [49]:
top_n_random = get_top_n_random(trainset, n=10)
print(top_n_random["457"])

['39', '36', '18', '13', '40', '23', '7', '1', '28', '4']


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457:

In [50]:
for item_id in top_n_random["457"]:
    print(info_videojuegos[item_id])

('The Witcher 3: Wild Hunt', 'Simulation')
('The Elder Scrolls V: Skyrim', 'Sports')
('Kingdom Hearts III', 'Action')
('Hades', 'Puzzle')
('Tomb Raider (2013)', 'Adventure')
('Pillars of Eternity II: Deadfire', 'Shooter')
('Cuphead', 'RPG')
('1000-Piece Puzzle', 'Fighting')
('Sid Meier’s Civilization VI', 'Party')
('Bioshock Infinite', 'Sports')


Finalmente, se evalúan las métricas de Random:

In [51]:
def evaluate_random_metrics(top_n_most_popular, k=10, reales=reales):
    precs, recs, ndcgs, hits, maps = [], [], [], [], []
    for u in usuarios_comunes:
        rec_k = top_n_random[u][:k]
        rel_u = reales[u]
        precs.append(precision_at_k(rec_k, rel_u))
        recs.append(recall_at_k(rec_k, rel_u))
        ndcgs.append(ndcg_at_k(rec_k, rel_u))
        hits.append(hit_score_at_k(rec_k, rel_u))
        maps.append(map_at_k(rec_k, rel_u))

    user_topk_eval = {u: top_n_random[u][:k] for u in usuarios_comunes}
    diversidad_k = diversity_at_k(user_topk_eval, info_videojuegos)

    return pd.DataFrame([{
        "Modelo": "Random",
        f"Precision@{k}": np.mean(precs),
        f"Recall@{k}": np.mean(recs),
        f"NDCG@{k}": np.mean(ndcgs),
        f"HitScore@{k}": np.mean(hits),
        f"F1@{k}": (2 * np.mean(precs) * np.mean(recs) / (np.mean(precs) + np.mean(recs))) if precs and recs and (np.mean(precs) + np.mean(recs)) > 0 else np.nan,
        f"MAP@{k}": np.mean(maps),
        f"Diversidad@{k}": diversidad_k,
        "Usuarios evaluados": len(usuarios_comunes)
    }])


In [52]:
display(evaluate_random_metrics(
    top_n_most_popular=top_n_random,
    k=10,
    reales=reales
    ))

,Modelo,Precision@10,Recall@10,NDCG@10,HitScore@10,F1@10,MAP@10,Diversidad@10,Usuarios evaluados
0,Random,0.02724,0.230984,0.109576,0.261649,0.048733,0.068614,6.677419,837


In [53]:
print("usuarios evaluados:", len(usuarios_comunes))
print("promedio relevantes por usuario:", np.mean([len(v) for v in reales.values()]))
print("total items posibles:", trainset.n_items)


usuarios evaluados: 837
promedio relevantes por usuario: 1.1911589008363201
total items posibles: 40


Se ve que el dataset tiene muy pocos ítems y muy pocos relevantes por usuario, por eso un azar uniforme tiene app. 25 % de probabilidad de acertar (10/40 = 0.25).

Las métricas top-10 son muy similares entre los métodos, lo que sugiere que el problema no permite gran discriminación entre modelos.
A mayor tamaño del catálogo o con más información de usuario-ítem, se esperaría que modelos basados en similitud (UserKNN, ItemKNN) y los baselines (MostPopular, Random) comiencen a diferenciarse más claramente.